<a href="https://colab.research.google.com/github/Statistics-Inho/Big-data-programming/blob/main/week04_5874941_%EC%B5%9C%EC%9D%B8%ED%98%B8%EC%9D%98.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4주차 과제
빅데이터프로그래밍 · 통계학과

**이름:** 최인호
**학번:** 5874941

## 제출 방법
모든 셀을 실행해 출력이 보이는 상태로 저장한 뒤 `week04_학번_이름.ipynb` 로 제출합니다.

GPU 문제는 `런타임 > 런타임 유형 변경 > T4 GPU` 로 바꾼 뒤 실행하세요.


In [48]:
import torch
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "/ device:", device)


torch 2.11.0+cu130 / device: cuda


## 문제 1. Tensor 5개 생성과 속성 확인표 (40점)
서로 다른 Tensor 5개를 만들고, 아래 항목을 담은 표를 출력하세요.

| 이름 | shape | dim | dtype | device |

각 Tensor를 왜 그렇게 만들었는지 주석으로 한 줄 적으세요.


In [49]:
import pandas as pd
import torch

# 1. 서로 다른 속성을 가진 5개의 텐서 생성
t1 = torch.tensor([1, 2, 3])  # 1차원 정수형 (CPU)
t2 = torch.tensor([1.0, 2.0, 3.0]) #실수형
t3 = torch.randn(4, 7)     #랜덤형
t4 = torch.zeros(10, 15)
t5 = torch.tensor([True, False, True], dtype=torch.bool)  # 1차원 불리언(Bool)형

# 텐서 리스트 및 이름 지정
tensors = [("Tensor 1", t1), ("Tensor 2", t2), ("Tensor 3", t3), ("Tensor 4", t4), ("Tensor 5", t5)]

# 표 작성을 위한 데이터 추출
data = []
for name, t in tensors:
    data.append({
        "이름": name,
        "shape": tuple(t.shape),
        "dim": t.dim(),
        "dtype": str(t.dtype).replace("torch.", ""),
        "device": str(t.device),
    })

# Pandas DataFrame을 이용한 표 출력
df = pd.DataFrame(data)
print(df.to_markdown(index=False))

| 이름     | shape    |   dim | dtype   | device   |
|:---------|:---------|------:|:--------|:---------|
| Tensor 1 | (3,)     |     1 | int64   | cpu      |
| Tensor 2 | (3,)     |     1 | float32 | cpu      |
| Tensor 3 | (4, 7)   |     2 | float32 | cpu      |
| Tensor 4 | (10, 15) |     2 | float32 | cpu      |
| Tensor 5 | (3,)     |     1 | bool    | cpu      |


## 문제 2. Tensor 연산 (20점)
(3, 4) 와 (4, 2) Tensor를 만들어 아래를 출력하세요.
- 행렬곱 결과와 그 shape
- 첫 번째 Tensor의 열별 평균과 행별 합계


In [50]:
# 답안
a = torch.tensor([3,4])
b = torch.tensor([4,2])
a@b



import torch

# 1. Tensor 생성 (3, 4) 및 (4, 2)
tensor_a = torch.tensor(
    [[1.0, 2.0, 3.0, 4.0], [5.0, 6.0, 7.0, 8.0], [9.0, 10.0, 11.0, 12.0]]
)

tensor_b = torch.tensor([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 8.0]])

# 2. 행렬곱 (Matrix Multiplication)
matmul_res = torch.matmul(tensor_a, tensor_b)

# 3. 첫 번째 Tensor의 연산
col_mean = tensor_a.mean(dim=0)  # 열별 평균 (각 열 내 행들을 평균)
row_sum = tensor_a.sum(dim=1)  # 행별 합계 (각 행 내 열들을 합산)

print("=== 1. 행렬곱 결과 및 Shape ===")
print("행렬곱 결과:\n", matmul_res)
print("Shape:", matmul_res.shape)

print("\n=== 2. 첫 번째 Tensor (3, 4) 연산 ===")
print("열별 평균 (dim=0):", col_mean)
print("행별 합계 (dim=1):", row_sum)

=== 1. 행렬곱 결과 및 Shape ===
행렬곱 결과:
 tensor([[ 50.,  60.],
        [114., 140.],
        [178., 220.]])
Shape: torch.Size([3, 2])

=== 2. 첫 번째 Tensor (3, 4) 연산 ===
열별 평균 (dim=0): tensor([5., 6., 7., 8.])
행별 합계 (dim=1): tensor([10., 26., 42.])


## 문제 3. CPU와 GPU 비교 (20점)
같은 크기의 행렬곱을 CPU와 GPU에서 각각 실행해 걸린 시간을 출력하고, 차이를 한두 줄로 설명하세요.
GPU를 쓸 수 없는 경우에는 CPU 결과와 그 이유를 적으세요.


In [51]:
# 답안
import time
import torch

# 대용량 행렬 크기 설정 (5000 x 5000)
size = 5000

# [CPU 연산 수행]
a_cpu = torch.randn(size, size, device="cpu")
b_cpu = torch.randn(size, size, device="cpu")

# 정확한 측정을 위한 워밍업(Warm-up)
_ = torch.matmul(a_cpu, b_cpu)

start_cpu = time.time()
result_cpu = torch.matmul(a_cpu, b_cpu)
end_cpu = time.time()
time_cpu = end_cpu - start_cpu

print(f"CPU 연산 시간: {time_cpu:.4f}초")

# [GPU 가용성 확인 및 연산 수행]
if torch.cuda.is_available():
    a_gpu = a_cpu.to("cuda")
    b_gpu = b_cpu.to("cuda")

    # 워밍업
    _ = torch.matmul(a_gpu, b_gpu)
    torch.cuda.synchronize()  # 비동기 처리 동기화

    start_gpu = time.time()
    result_gpu = torch.matmul(a_gpu, b_gpu)
    torch.cuda.synchronize()  # GPU 연산 종료 대기
    end_gpu = time.time()
    time_gpu = end_gpu - start_gpu

    print(f"GPU 연산 시간: {time_gpu:.4f}초")
    print(f"속도 차이: GPU가 약 {time_cpu / time_gpu:.2f}배 빠름")
else:
    print("\n[GPU 사용 불가능 환경]")
    print(
        "GPU를 사용할 수 없어 CUDA 지원 코드를 통한 시간 측정이 불가능합니다."
    )

CPU 연산 시간: 2.3329초
GPU 연산 시간: 0.0819초
속도 차이: GPU가 약 28.47배 빠름


## 문제 4. 자동미분 (20점)
아래 두 가지를 하세요.
1. $y = 2x^3 + 5x$ 에서 $x=3$ 일 때의 도함수를 자동미분으로 구하고, 손으로 구한 값과 비교하세요.
2. 자기가 만든 수식 하나에 대해 미분값을 구하고, 손 계산과 일치하는지 확인하세요.


In [52]:
# 답안
import torch

# 1. 미분 대상 변수 텐서 생성 (requires_grad=True 필수)
x = torch.tensor(3.0, requires_grad=True)

# 2. 연산 그래프 생성
y = 2 * (x**3) + 5 * x

# 3. 역전파(Backpropagation) 수행 - 자동미분 계산
y.backward()

# 4. x = 3.0에서의 미분값 dy/dx 확인 (x.grad)
print(f"입력값 x: {x.item()}")
print(f"함수값 y: {y.item()}")
print(f"자동미분 결과 (dy/dx): {x.grad.item()}")

입력값 x: 3.0
함수값 y: 69.0
자동미분 결과 (dy/dx): 59.0


In [53]:
# 답안
import torch

# 1. 미분 대상 변수 텐서 생성 (requires_grad=True 필수)
x = torch.tensor(3.0, requires_grad=True)

# 2. 연산 그래프 생성
y = 3*(x**2)+7*x+7

# 3. 역전파(Backpropagation) 수행 - 자동미분 계산
y.backward()

# 4. x = 3.0에서의 미분값 dy/dx 확인 (x.grad)
print(f"입력값 x: {x.item()}")
print(f"함수값 y: {y.item()}")
print(f"자동미분 결과 (dy/dx): {x.grad.item()}")

입력값 x: 3.0
함수값 y: 55.0
자동미분 결과 (dy/dx): 25.0
